In [ ]:
import os
import requests
import pandas as pd
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import sys
!{sys.executable} -m pip install mygene

In [ ]:
pip install pyarrow

In [ ]:
import os
import sys
from pathlib import Path

PREP_DIR = Path.cwd()  # run with CWD = additional_data_source/opentarget/
sys.path.insert(0, str(PREP_DIR.parent))
from release_paths import resolve_release_root, primary_data_dir

RELEASE_ROOT = resolve_release_root()
DATA_DIR = primary_data_dir(RELEASE_ROOT)
OPENTARGET_OUT = DATA_DIR / "disgenet" / "OpenTarget" / "OpenTarget_disease_protein_associations.csv"
OPENTARGET_OUT.parent.mkdir(parents=True, exist_ok=True)
data_path = str(DATA_DIR) + os.sep


In [ ]:
# Run from additional_data_source/opentarget/ — inputs here; outputs -> primary_data_prep/data/


In [ ]:
# ==== 1. Configuration ====
BASE_URL = "https://ftp.ebi.ac.uk/pub/databases/opentargets/platform/25.09/output/association_overall_direct/"
SAVE_DIR = "opentargets_associations"

os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# ==== 2. List *.parquet files in GCS folder ====
def list_parquet_files(url):
    r = requests.get(url)
    soup = BeautifulSoup(r.text, "html.parser")
    files = []

    for link in soup.find_all("a"):
        href = link.get("href")
        if href.endswith(".parquet"):
            files.append(urljoin(url, href))
    return files

files = list_parquet_files(BASE_URL)
print(f"Found {len(files)} files")

In [ ]:
# ==== 3. Download each file ====
def download_file(url, save_dir):
    name = url.split("/")[-1]
    path = os.path.join(save_dir, name)
    print(path)
    if not os.path.exists(path):
        print("Downloading:", name)
        r = requests.get(url)
        with open(path, "wb") as f:
            f.write(r.content)
    else:
        print("Already exists:", name)

    return path

# local_files = [download_file(f, SAVE_DIR) for f in files]

In [ ]:
# local_files

In [ ]:
ls inputs/opentargets_associations

In [ ]:
local_files = os.listdir("inputs/opentargets_associations")
local_files

In [ ]:
path = "inputs/opentargets_associations"

In [ ]:
ls

In [ ]:
# ==== 4. Read and merge all parquet files ====
# dfs = [pd.read_parquet(path + "/"+ f) for f in local_files]
# full_df = pd.concat(dfs, ignore_index=True)

# print("Final dataset shape:", full_df.shape)

# # Save to one file
# full_df.to_csv("OpenTargets_associations_merged.csv")

full_df = pd.read_csv("OpenTargets_associations_merged.csv")


In [ ]:
full_df.head(3)

In [ ]:
full_df.evidenceCount.describe()

In [ ]:
full_df.score.min(), full_df.score.max()

In [ ]:
len(full_df), len(full_df.diseaseId.unique())

In [ ]:
full_df.head(3)

In [ ]:
opentargetThreshold = 0.1
full_df = full_df[full_df.score>=opentargetThreshold] #change here for difference comparison between score =0.1 and 0.3

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Histogram (log-scale y-axis to show long tail)
axes[0].hist(full_df['evidenceCount'], bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_yscale('log')
axes[0].set_xlabel('Evidence Count')
axes[0].set_ylabel('Frequency (log scale)')
axes[0].set_title('Evidence Count Distribution (log y)')

# 2. ECDF — cumulative % for threshold choice
sorted_counts = np.sort(full_df['evidenceCount'])
ecdf = np.arange(1, len(sorted_counts)+1) / len(sorted_counts)
axes[1].plot(sorted_counts, ecdf * 100, color='darkorange')
axes[1].axvline(x=2, color='red', linestyle='--', alpha=0.7, label='count=2')
axes[1].axvline(x=3, color='green', linestyle='--', alpha=0.7, label='count=3')
axes[1].set_xlabel('Evidence Count')
axes[1].set_ylabel('Cumulative %')
axes[1].set_title('ECDF of Evidence Count')
axes[1].legend()
axes[1].set_xlim(0, 20)  # zoom to head

# 3. Fix: use seaborn boxplot instead of pandas boxplot
import seaborn as sns
count_cap = min(full_df['evidenceCount'].quantile(0.95), 20)
df_plot = full_df[full_df['evidenceCount'] <= count_cap].copy()
df_plot['evidenceCount'] = df_plot['evidenceCount'].astype(int)

sns.boxplot(data=df_plot, x='evidenceCount', y='score', ax=axes[2], color='steelblue')
axes[2].set_xlabel('Evidence Count')
axes[2].set_ylabel('Score')
axes[2].set_title('Score by Evidence Count')

plt.suptitle('OpenTargets - Evidence Count Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary stats
print(full_df['evidenceCount'].describe())
print(f"\nProportion with count=1:  {(full_df['evidenceCount']==1).mean():.1%}")
print(f"Proportion with count>=2: {(full_df['evidenceCount']>=2).mean():.1%}")
print(f"Proportion with count>=3: {(full_df['evidenceCount']>=3).mean():.1%}")

In [ ]:
full_df_bkup = full_df
full_df = full_df[full_df.score>=0.1]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Histogram (log-scale y-axis to show long tail)
axes[0].hist(full_df['evidenceCount'], bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_yscale('log')
axes[0].set_xlabel('Evidence Count')
axes[0].set_ylabel('Frequency (log scale)')
axes[0].set_title('Evidence Count Distribution (log y)')

# 2. ECDF — cumulative % for threshold choice
sorted_counts = np.sort(full_df['evidenceCount'])
ecdf = np.arange(1, len(sorted_counts)+1) / len(sorted_counts)
axes[1].plot(sorted_counts, ecdf * 100, color='darkorange')
axes[1].axvline(x=2, color='red', linestyle='--', alpha=0.7, label='count=2')
axes[1].axvline(x=3, color='green', linestyle='--', alpha=0.7, label='count=3')
axes[1].set_xlabel('Evidence Count')
axes[1].set_ylabel('Cumulative %')
axes[1].set_title('ECDF of Evidence Count')
axes[1].legend()
axes[1].set_xlim(0, 20)  # zoom to head

# 3. Fix: use seaborn boxplot instead of pandas boxplot
import seaborn as sns
count_cap = min(full_df['evidenceCount'].quantile(0.95), 20)
df_plot = full_df[full_df['evidenceCount'] <= count_cap].copy()
df_plot['evidenceCount'] = df_plot['evidenceCount'].astype(int)

sns.boxplot(data=df_plot, x='evidenceCount', y='score', ax=axes[2], color='steelblue')
axes[2].set_xlabel('Evidence Count')
axes[2].set_ylabel('Score')
axes[2].set_title('Score by Evidence Count')

plt.suptitle('OpenTargets - Evidence Count Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary stats
print(full_df['evidenceCount'].describe())
print(f"\nProportion with count=1:  {(full_df['evidenceCount']==1).mean():.1%}")
print(f"Proportion with count>=2: {(full_df['evidenceCount']>=2).mean():.1%}")
print(f"Proportion with count>=3: {(full_df['evidenceCount']>=3).mean():.1%}")

In [ ]:
full_df['evidenceCount'].max()

In [ ]:
len(full_df), len(full_df.diseaseId.unique())

In [ ]:
disease_df = pd.read_parquet("inputs/disease.parquet")
len(disease_df)

In [ ]:
disease_df.head(4)

In [ ]:
# !tail -n +6  Homo_sapiens.GRCh38.115.gtf > Homo_sapiens.GRCh38.115.no.header.gtf

In [ ]:
# !head -n5 Homo_sapiens.GRCh38.115.no.header.gtf

In [ ]:
# gtf = pd.read_csv("Homo_sapiens.GRCh38.115.no.header.gtf", sep="\t", header=None)
# gtf.columns = ["col0", "col1", "col2", "col3", "col4", "col5", "col6", "col7", "col8"]
# gtf.head(3)

In [ ]:
# gtf = gtf[(gtf.col2=="gene")&(gtf.col8.str.contains("gene_name "))]
# len(gtf)

In [ ]:
# gtf.head()

In [ ]:
# with open("homo.sapien.txt", "w") as f:
#     for value in gtf["col8"]:
#         pairs = value.split(";")
#         for pair in pairs:
#             f.write(str(value) + "\n")

In [ ]:
# !head -n10 homo.sapien.txt # we still take genes with gene_biotype "processed_pseudogene" as this is just for annotation

In [ ]:
# temp = []
# with open("homo.sapien.txt", "r") as f:
#     lines = f.readlines()
#     for line in lines:
#         pairs = line[:-1].split("; ")
#         t = []
#         for pair in pairs:
#             t.append(pair.split(" ")[1][1:-1])
#         t[-1]= t[-1][:-1]
#         temp.append(t)
# geneAnnotation = pd.DataFrame(temp, columns=["gene_id","gene_version","gene_name","gene_source","gene_biotype"])
# geneAnnotation = geneAnnotation[geneAnnotation.gene_biotype == "protein_coding"]
# geneAnnotation = geneAnnotation.drop_duplicates(subset=["gene_id", "gene_name"])
# geneAnnotation

In [ ]:
print(len(full_df), len(full_df.targetId.unique()))
full_df.head(2)

In [ ]:
unique_targets = full_df['targetId'].dropna().astype(str).unique().tolist()
# remove any non-ENSG (if your ids include ENSP etc.)
ensgs = [t for t in unique_targets if t.startswith("ENSG")]
len(ensgs)

In [ ]:
#time consuming -run once
dict = {}
for i, ensg in enumerate(ensgs):
    url = f"https://rest.ensembl.org/lookup/id/{ensg}?content-type=application/json"
    try:
        r = requests.get(url, headers={"Content-Type":"application/json"})
        r.raise_for_status()
        js = r.json()
        symbol = js.get("display_name")
        dict[ensg] = symbol
    except Exception as e:
        # Fallback: store None or "NA" and print warning
        dict[ensg] = None
        print(f"Warning: failed to fetch {ensg} — {e}")
    if (i)%100==0:
        print("progressing at i = ",i)

In [ ]:
len(dict.keys())

In [ ]:
with open("20260417-EnsemblID-Genename.csv", "w") as f:
    for key in dict.keys():
        v = dict.get(key)
        # print(v, " ", type(v))
        if v!=None:
            f.write(key+","+v+"\n")

In [ ]:
temp = pd.read_csv("20260417-EnsemblID-Genename.csv", header=None)
temp.columns = ["EnsemblID", "Genename"]
print(len(temp))
temp.head(3)

In [ ]:
full_df.head(2)

In [ ]:
disease_df.head(2)

In [ ]:
rs = pd.merge(full_df, temp, left_on="targetId", right_on="EnsemblID", how="inner")
rs.head(2)

In [ ]:
rs = pd.merge(rs, disease_df, left_on="diseaseId", right_on="id", how="inner")
rs.head(2)

In [ ]:
rs.columns

In [ ]:
len(rs)

In [ ]:
rs = rs[["Genename", "score", "EnsemblID", "name","dbXRefs"]]
rs.head(2)

In [ ]:
import pandas as pd
import numpy as np
import ast

def parse_pairs(cell):

    # 1️⃣ Actual NaN
    if cell is None:
        return {}

    # 2️⃣ numpy array
    if isinstance(cell, np.ndarray):
        cell = cell.tolist()

    # 3️⃣ Actual list
    if isinstance(cell, list):
        items = cell

    # 4️⃣ string
    elif isinstance(cell, str):
        cell = cell.strip()

        # empty string
        if cell == "":
            return {}

        # string representation of a list
        if cell.startswith("[") and cell.endswith("]"):
            try:
                items = ast.literal_eval(cell)
            except:
                return {}
        else:
            # string like MONDO:..., NANDO:...
            items = [x.strip() for x in cell.split(",")]

    else:
        return {}

    # 5️⃣ Parse each item
    result = {}
    for item in items:
        if isinstance(item, str) and ":" in item:
            name, value = item.split(":", 1)
            result[name.strip()] = value.strip()

    return result


parsed_series = rs["dbXRefs"].apply(parse_pairs)

# all_names = set()
# for d in parsed_series:
#     all_names.update(d.keys())

# print("All names:", all_names)

expanded_df = pd.DataFrame(parsed_series.tolist())

df_final = pd.concat([rs.drop(columns=["dbXRefs"]), expanded_df], axis=1)


In [ ]:
df_final.head(3)

In [ ]:
len(df_final)

In [ ]:
df_final.columns

In [ ]:
df_final = df_final[['Genename', 'score', 'EnsemblID', 'name', 'UMLS']]

In [ ]:
df_final.head(5)

In [ ]:
df_final = df_final.drop_duplicates()
len(df_final)

In [ ]:
df_final = df_final.dropna()
len(df_final)

In [ ]:
len(df_final.name.unique())

In [ ]:
len(df_final.EnsemblID.unique())

In [ ]:
data_path = os.environ.get("PRIMEKG_ROOT", "..") + "/datasets/data/"  # set PRIMEKG_ROOT for full rebuild

In [ ]:
df_disgenet = pd.read_csv(data_path+'disgenet/Authors-curated_gene_disease_associations.tsv', sep='\t', low_memory=False)
df_disgenet = df_disgenet.astype({'geneId':int}).astype({'geneId':str})
print("len df_disgenet:", len(df_disgenet))
# print("columns df_disgenet:", df_disgenet.columns)
df_disgenet.head(3)

In [ ]:
df_disgenet.diseaseName.unique()[:5]

In [ ]:
df_disgenet.diseaseName = df_disgenet.diseaseName.str.lower()

In [ ]:
df_disgenet.head(2)

In [ ]:
df_final.head(2)

In [ ]:
#now calculate the overalap between df_disgenet (score >=0.3) and disgenet (df_final, no score threshold)
df_disgenet_simple = df_disgenet[["geneSymbol", "diseaseId", "score"]]
opentarget_simple = df_final[["Genename", "UMLS", "score"]]
opentarget_simple.columns = ["geneSymbol", "diseaseId", "score"]

In [ ]:
print(len(opentarget_simple))
print(len(opentarget_simple[opentarget_simple.score>=0.3]))
print(len(df_disgenet_simple[df_disgenet_simple.score>=0.3]))

In [ ]:
78853/3114073

In [ ]:
opentarget_simple.score.describe()

In [ ]:
df_disgenet_simple.score.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 16,
})

plt.figure(figsize=(8,5))

sns.histplot(df_disgenet_simple["score"], 
             label=f"DisGeNET (>=0.3)", 
             kde=True, 
             stat="density", 
             bins=50)

sns.histplot(opentarget_simple["score"], 
             label=f"OpenTargets (>={opentargetThreshold})", 
             kde=True, 
             stat="density", 
             bins=50)

plt.legend()
plt.xlabel("Score")
plt.title("Score distribution comparison")
plt.show()

In [ ]:
# plt.figure(figsize=(8,5))

sns.kdeplot(df_disgenet_simple["score"], label="DisGeNET (>=0.3)", cut=0)
sns.kdeplot(opentarget_simple["score"], label=f"OpenTargets (>={opentargetThreshold})", cut=0)

plt.legend()
plt.xlabel("Score")
plt.title("Score distribution (all associations)")
# Save
plt.savefig(f"score_distribution_all{opentargetThreshold}.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df_overlap = df_disgenet_simple.merge(
    opentarget_simple,
    on=["geneSymbol", "diseaseId"],
    suffixes=("_disgenet", "_opentarget")
)
print(len(df_overlap))
df_overlap.head()

In [ ]:
df_plot = pd.concat([
    df_overlap[["score_disgenet"]].rename(columns={"score_disgenet": "score"})
        .assign(source="DisGeNET (>=0.3)"),
    df_overlap[["score_opentarget"]].rename(columns={"score_opentarget": "score"})
        .assign(source=f"OpenTargets (>={opentargetThreshold})")
])

plt.figure(figsize=(8,5))

sns.kdeplot(data=df_plot, x="score", hue="source", common_norm=False, cut=0)

plt.title("Score distribution (overlapping associations)")
plt.xlabel("Score")
# Save
plt.savefig(f"score_distribution_overlapping{opentargetThreshold}.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df_overlap

In [ ]:
from scipy.stats import gaussian_kde

kde1 = gaussian_kde(df_overlap["score_disgenet"])
kde2 = gaussian_kde(df_overlap["score_opentarget"])

x = np.linspace(-0.5, 1.5, 2000)

print(np.trapz(kde1(x), x))  # ~1
print(np.trapz(kde2(x), x))  # ~1

In [ ]:


# --- Step 1: find overlapping pairs ---
df_overlap = df_disgenet_simple.merge(
    opentarget_simple,
    on=["geneSymbol", "diseaseId"],
    how="inner"
)

# --- Step 2: find non-overlapping pairs ---
df_disgenet_non = df_disgenet_simple.merge(
    df_overlap[["geneSymbol", "diseaseId"]],
    on=["geneSymbol", "diseaseId"],
    how="left",
    indicator=True
).query('_merge == "left_only"').drop(columns="_merge")

df_opentarget_non = opentarget_simple.merge(
    df_overlap[["geneSymbol", "diseaseId"]],
    on=["geneSymbol", "diseaseId"],
    how="left",
    indicator=True
).query('_merge == "left_only"').drop(columns="_merge")

# --- Step 3: reshape to long format ---
df_plot = pd.concat([
    df_disgenet_non[["score"]].assign(source="DisGeNET (>=0.3)"),
    df_opentarget_non[["score"]].assign(source=f"OpenTargets (>={opentargetThreshold})")
])

# --- Step 4: plot ---
plt.figure(figsize=(8,5))

sns.kdeplot(data=df_plot, x="score", hue="source", common_norm=False, cut=0)

plt.xlabel("Score")
plt.title("Score distribution (non-overlapping associations)")

# Save
plt.savefig(f"score_distribution_non_overlapping{opentargetThreshold}.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import mygene
import pandas as pd

mg = mygene.MyGeneInfo()

genes = df_final.Genename.unique()

result = mg.querymany(
    genes,
    scopes="symbol",
    fields="entrezgene",
    species="human"
)

df_ids = pd.DataFrame(result)
print(df_ids[["query", "entrezgene"]])


In [ ]:
df_ids = df_ids[['query', 'entrezgene']]

In [ ]:
df_ids.columns = ["Genename", "NCBI_ID"]

In [ ]:
df_disgenet_disease = df_disgenet.diseaseName.unique()
df_final_disease = df_final.name.unique()
len([x for x in df_final_disease if not x in df_disgenet_disease])

In [ ]:
df_disgenet_gene = df_disgenet.geneSymbol.unique()
df_final_gene = df_final.Genename.unique()
len([x for x in df_final_gene if not x in df_disgenet_gene])

In [ ]:
df_disgenet["pairs"] = df_disgenet["geneSymbol"] + "_" + df_disgenet["diseaseName"]
df_final["pairs"] = df_final["Genename"] + "_" + df_final["name"]
df_final.head(3)

In [ ]:
df_disgenet_pairs = df_disgenet.pairs.unique()
df_final_pairs = df_final.pairs.unique()
new_pairs = [p for p in df_final_pairs if not p in df_disgenet_pairs]
rows_for_adding = df_final[df_final.pairs.isin(new_pairs)]
len(rows_for_adding), len(df_final)

In [ ]:
new_pairs

In [ ]:
rows_for_adding.head(3)

In [ ]:
df_ids.head(3)

In [ ]:
final_rs = pd.merge(rows_for_adding, df_ids, left_on="Genename", right_on="Genename", how="inner")

In [ ]:
final_rs.head(3)

In [ ]:
final_rs.columns

In [ ]:
# geneId → renamed to x_id (gene/protein ID)
# geneSymbol → renamed to x_name (gene/protein name)
# diseaseId → merge with df_umls_mondo (not in final kg.csv)
# diseaseType → used to filter:

final_rs.columns = ['x_name', 'score', 'EnsemblID', 'diseaseName', 'diseaseId', 'pairs', "x_id"]

In [ ]:
final_rs.head(10)

In [ ]:
final_rs.to_csv("OpenTarget_disease_protein_associations.csv")

In [ ]:
!pwd